# 1. ReviewLens GPT — 01 · Preprocessing & Validasi Grouped Split

## Tujuan eksperimen

Notebook ini menjalankan pipeline persiapan data **yang sudah ada di repository**
(`scripts/prepare_data.py`) pada dataset asli, lalu memvalidasi hasil
*grouped split* sebelum training Mini-Transformer.

Hal yang diperiksa:

1. Ulasan dengan teks identik setelah `clean_text` **tidak** muncul di dua split
   berbeda (tidak ada kebocoran train → val/test).
2. Kelompok ulasan dengan label sentimen yang saling bertentangan
   (*conflict groups*, keputusan A2) dibuang seluruhnya, dan jumlahnya tercatat.
3. Ulasan identik dengan sentimen sama (keputusan B1) tetap dipertahankan dalam satu split.
4. Proporsi train/val/test mendekati 80/10/10, dan oversampling negatif hanya terjadi di train.

## Sumber dataset

- File: `Processed Dataset.csv` (ulasan Lazada Indonesia, kategori elektronik).
- Dipasang **manual** lewat *Add Input* di Kaggle Notebook, lalu dicari otomatis di bawah `/kaggle/input/`.
- Dataset diperlakukan **read-only**. CSV asli tidak pernah ditulis, dan hash SHA-256-nya dicek sebelum dan sesudah eksekusi.

## Alur pipeline (sesuai `INTERFACES.md` §4.1 dan §13)

`load_reviews` → `clean_text` → buang ulasan kosong → `split_data_with_stats`
(grouped by `clean_text(review)`, conflict groups dibuang, stratified per sentimen, 80/10/10, seed 42)
→ silver labeling (`label_review`) → oversampling negatif **hanya di train** → tulis JSONL.

## Output yang diharapkan

| File | Isi |
|---|---|
| `/kaggle/working/data/processed_grouped_validation/train.jsonl` | split train (negatif sudah di-oversample) |
| `/kaggle/working/data/processed_grouped_validation/val.jsonl` | split validation |
| `/kaggle/working/data/processed_grouped_validation/test.jsonl` | split test |
| `/kaggle/working/prepare_data_validation_stdout.txt` | log lengkap `prepare_data.py` |
| `/kaggle/working/prepare_data_validation_summary.json` | ringkasan eksperimen dan hasil semua pengecekan |

**Di luar cakupan notebook ini:** training model, pembangunan tokenizer/vocabulary,
dan perubahan source code. Vocabulary dibangun di tahap berikutnya **hanya** dari `train.jsonl`.

> Semua angka di notebook ini berasal dari eksekusi nyata pada dataset yang dipasang. Tidak ada data sintetis.

## 2. Mengambil source code (branch `main` terbaru)

Repository di-clone ke `/kaggle/working/reviewlens-mini-gpt`.
**Internet harus ON** (panel kanan → *Settings* → *Internet*).

Jika folder sudah ada (misalnya notebook dijalankan ulang), sel ini **tidak menghapus apa pun**:

- memastikan folder adalah repo git dengan `origin` yang benar;
- **berhenti dengan error** jika ada perubahan lokal (tidak menimpa pekerjaan apa pun);
- jika bersih: `git fetch origin main`, lalu memindahkan branch lokal `main` ke `origin/main`.

Hanya operasi baca dan operasi lokal yang dipakai. Tidak ada `add`, `commit`, `push`, atau `merge`,
dan remote tidak diubah.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nidahijriaa/reviewlens-mini-gpt.git"
BRANCH = "main"
WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "reviewlens-mini-gpt"


def run_git(*args, cwd=None):
    """Run a git command and return stdout; raise a clear error on failure."""
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"`git {' '.join(args)}` gagal (exit {res.returncode}):\n{res.stderr.strip()}"
        )
    return res.stdout.strip()


def _norm_url(url):
    url = url.strip().rstrip("/")
    return url[:-4] if url.endswith(".git") else url


if shutil.which("git") is None:
    raise RuntimeError("Perintah `git` tidak tersedia di environment ini.")
if not WORKING_DIR.is_dir():
    raise FileNotFoundError(f"{WORKING_DIR} tidak ada. Notebook ini dirancang untuk Kaggle Notebook.")

if not REPO_DIR.exists():
    print(f"Clone {REPO_URL} (branch {BRANCH}) -> {REPO_DIR}")
    try:
        run_git("clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR))
    except RuntimeError as exc:
        raise RuntimeError(
            "Clone gagal. Pastikan Internet ON di Settings notebook Kaggle, lalu jalankan ulang sel ini.\n"
            + str(exc)
        ) from exc
else:
    print(f"Repository sudah ada di {REPO_DIR}; memeriksa sebelum memperbarui.")
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(
            f"{REPO_DIR} ada tetapi bukan repository git. Periksa isinya; "
            "hapus manual jika memang tidak dipakai, lalu jalankan ulang sel ini."
        )
    origin = run_git("remote", "get-url", "origin", cwd=REPO_DIR)
    if _norm_url(origin) != _norm_url(REPO_URL):
        raise RuntimeError(f"origin = {origin!r}, bukan {REPO_URL!r}. Tidak diperbarui otomatis.")
    dirty = run_git("status", "--porcelain", cwd=REPO_DIR)
    if dirty:
        raise RuntimeError(
            "Ada perubahan lokal di repository; tidak ditimpa otomatis:\n" + dirty
            + "\nSimpan/hapus perubahan tersebut secara manual, lalu jalankan ulang sel ini."
        )
    run_git("fetch", "origin", BRANCH, cwd=REPO_DIR)
    # Local-only: point local `main` at the freshly fetched origin/main (no merge, no push).
    run_git("checkout", "-B", BRANCH, f"origin/{BRANCH}", cwd=REPO_DIR)

head = run_git("rev-parse", "HEAD", cwd=REPO_DIR)
remote_head = run_git("rev-parse", f"origin/{BRANCH}", cwd=REPO_DIR)
current_branch = run_git("rev-parse", "--abbrev-ref", "HEAD", cwd=REPO_DIR)
if head != remote_head or current_branch != BRANCH:
    raise RuntimeError(
        f"HEAD ({current_branch} @ {head[:7]}) tidak sama dengan origin/{BRANCH} ({remote_head[:7]})."
    )
REPO_COMMIT = head
print("Branch :", current_branch)
print("Commit :", run_git("log", "-1", "--format=%H%n         %s (%ci)", cwd=REPO_DIR))

## 3. Menemukan dataset Kaggle (Add Input)

Dataset **tidak diunduh** lewat Kaggle API atau `kagglehub`, dan tidak disalin ke repository.
Pasang dataset secara manual: panel kanan → **Add Input** → pilih dataset yang berisi
`Processed Dataset.csv`.

Sel ini mencari file bernama persis `Processed Dataset.csv` di seluruh `/kaggle/input/`
memakai `pathlib`, tanpa menebak struktur folder:

- **0 kandidat** → error dengan petunjuk Add Input.
- **> 1 kandidat** → semua kandidat ditampilkan dan sel berhenti. Isi `CSV_PATH_OVERRIDE`
  dengan salah satu path yang ditampilkan, lalu jalankan ulang. File tidak dipilih diam-diam.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
CSV_NAME = "Processed Dataset.csv"
CSV_PATH_OVERRIDE = None   # isi HANYA jika ditemukan lebih dari satu kandidat, mis. "/kaggle/input/.../Processed Dataset.csv"

ADD_INPUT_HELP = (
    f"Tambahkan dataset yang berisi '{CSV_NAME}' lewat panel kanan Kaggle: "
    "Add Input -> pilih dataset -> Add. Setelah itu jalankan ulang sel ini."
)

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f"{INPUT_ROOT} tidak ditemukan. {ADD_INPUT_HELP}")

candidates = sorted(p for p in INPUT_ROOT.rglob(CSV_NAME) if p.is_file())
print(f"Kandidat '{CSV_NAME}' di bawah {INPUT_ROOT}: {len(candidates)}")
for i, p in enumerate(candidates, 1):
    print(f"  [{i}] {p}  ({p.stat().st_size / 1e6:.2f} MB)")

if CSV_PATH_OVERRIDE is not None:
    chosen = Path(CSV_PATH_OVERRIDE)
    if chosen.resolve() not in {p.resolve() for p in candidates}:
        raise ValueError(
            f"CSV_PATH_OVERRIDE = {chosen} bukan salah satu kandidat yang ditemukan di atas."
        )
    CSV_PATH = chosen
    print(f"\nDipilih eksplisit lewat CSV_PATH_OVERRIDE: {CSV_PATH}")
elif not candidates:
    raise FileNotFoundError(f"File '{CSV_NAME}' tidak ditemukan di bawah {INPUT_ROOT}. {ADD_INPUT_HELP}")
elif len(candidates) > 1:
    raise RuntimeError(
        f"Ditemukan {len(candidates)} file '{CSV_NAME}' (lihat daftar di atas). "
        "File tidak dipilih otomatis: isi CSV_PATH_OVERRIDE dengan path yang benar, lalu jalankan ulang sel ini."
    )
else:
    CSV_PATH = candidates[0]
    print(f"\nDataset: {CSV_PATH}")


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


CSV_SHA256_BEFORE = sha256_file(CSV_PATH)
print("SHA-256 (sebelum eksekusi):", CSV_SHA256_BEFORE)

## 4. Verifikasi source code dan dataset

- File script dan modul `src/reviewlens/` yang dipakai `prepare_data.py` harus ada.
- Modul di-import dari repository yang baru di-clone (bukan dari lokasi lain).
- Kolom wajib dibaca dari `preprocess.COLUMN_MAP`, bukan ditulis ulang di notebook.
- CSV hanya **dibaca** (`pd.read_csv`) untuk inspeksi; tidak ada penulisan.

In [ ]:
import importlib
import importlib.util

import numpy as np
import pandas as pd

REQUIRED_FILES = [
    "scripts/prepare_data.py",
    "src/reviewlens/__init__.py",
    "src/reviewlens/config.py",
    "src/reviewlens/preprocess.py",
    "src/reviewlens/labeling.py",
    "src/reviewlens/dataset.py",
    "INTERFACES.md",
]
missing_files = [f for f in REQUIRED_FILES if not (REPO_DIR / f).is_file()]
if missing_files:
    raise FileNotFoundError(f"File berikut tidak ada di {REPO_DIR}: {missing_files}")
print("Semua file yang dibutuhkan tersedia:")
for f in REQUIRED_FILES:
    print("  ok", f)

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
# Re-run safety: drop any previously imported copy so the cloned code is used.
for name in [m for m in sys.modules if m == "reviewlens" or m.startswith("reviewlens.")]:
    del sys.modules[name]

preprocess = importlib.import_module("reviewlens.preprocess")
if Path(preprocess.__file__).resolve() != (SRC_DIR / "reviewlens" / "preprocess.py").resolve():
    raise ImportError(f"reviewlens.preprocess di-import dari lokasi tak terduga: {preprocess.__file__}")
from reviewlens.preprocess import COLUMN_MAP, OPTIONAL_COLUMNS, clean_text, load_reviews

# Load scripts/prepare_data.py as a module (for its constants and defaults only).
_spec = importlib.util.spec_from_file_location("prepare_data", REPO_DIR / "scripts" / "prepare_data.py")
prep_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(prep_mod)
SPLITS = tuple(prep_mod.SPLITS)
RECORD_FIELDS = tuple(prep_mod.RECORD_FIELDS)
print("\nreviewlens.preprocess :", preprocess.__file__)
print("SPLITS                :", SPLITS)
print("RECORD_FIELDS (JSONL) :", RECORD_FIELDS)

# --- Dataset (read-only inspection) ------------------------------------------
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")   # same encoding as load_reviews
raw.columns = [str(c).strip() for c in raw.columns]  # in memory only
N_RAW_ROWS = len(raw)
RAW_COLUMNS = list(raw.columns)

missing_cols = [c for c in COLUMN_MAP if c not in RAW_COLUMNS]
if missing_cols:
    raise ValueError(f"Kolom wajib (preprocess.COLUMN_MAP) tidak ada di CSV: {missing_cols}")

print(f"\nJumlah baris CSV : {N_RAW_ROWS:,}")
print(f"Jumlah kolom     : {len(RAW_COLUMNS)}")
print("Kolom            :", RAW_COLUMNS)
print("\nPemetaan kolom (preprocess.COLUMN_MAP):")
for src, dst in COLUMN_MAP.items():
    print(f"  {src:16s} -> {dst}")
for c in OPTIONAL_COLUMNS:
    print(f"  {c:16s} -> {'ada (opsional, tidak dipakai di MVP)' if c in RAW_COLUMNS else 'tidak ada (opsional)'}")

used = list(COLUMN_MAP) + [c for c in OPTIONAL_COLUMNS if c in RAW_COLUMNS]
print("\nTipe data & nilai kosong pada kolom yang dipakai:")
print(pd.DataFrame({"dtype": raw[used].dtypes.astype(str), "n_null": raw[used].isna().sum()}).to_string())

print("\nDistribusi sentiment_label (mentah):")
print(raw["sentiment_label"].value_counts(dropna=False).to_string())
print("\nDistribusi rating (mentah):")
print(raw["rating"].value_counts(dropna=False).sort_index().to_string())
n_words = raw["reviewContent"].fillna("").astype(str).str.split().str.len()
print("\nPanjang reviewContent (kata):")
print(n_words.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_string())
print("\nContoh 5 baris pertama (kolom yang dipakai):")
print(raw[used].head().to_string())

## 5. Verifikasi command line `prepare_data.py`

1. `--help` harus berjalan dan memuat semua opsi yang dipakai implementasi saat ini.
2. Nilai default CLI dibaca dari **parser asli** script: `main()` dipanggil dengan
   `prepare()` diganti sementara oleh fungsi penangkap argumen, sehingga tidak ada
   pemrosesan atau penulisan file. Hasilnya dibandingkan dengan default `prepare()`
   dan dengan nilai yang didokumentasikan (`val_frac=0.1`, `test_frac=0.1`, `seed=42`).

Notebook memakai **default ini apa adanya** dan hanya mengisi `--csv` dan `--out-dir`.

In [ ]:
import inspect

PREPARE_SCRIPT = REPO_DIR / "scripts" / "prepare_data.py"
CHILD_ENV = {**os.environ, "PYTHONIOENCODING": "utf-8"}   # reviews may contain emoji
help_res = subprocess.run(
    [sys.executable, str(PREPARE_SCRIPT), "--help"], cwd=REPO_DIR, capture_output=True,
    text=True, encoding="utf-8", env=CHILD_ENV,
)
print(help_res.stdout)
if help_res.returncode != 0:
    raise RuntimeError(f"`prepare_data.py --help` gagal (exit {help_res.returncode}):\n{help_res.stderr}")

EXPECTED_OPTIONS = ["--csv", "--out-dir", "--val-frac", "--test-frac", "--seed", "--neg-oversample"]
missing_opts = [o for o in EXPECTED_OPTIONS if o not in help_res.stdout]
if missing_opts:
    raise RuntimeError(f"Opsi berikut tidak ada di --help: {missing_opts}. Implementasi berubah; periksa script.")
print("Semua opsi yang diharapkan tersedia:", EXPECTED_OPTIONS)


class _ArgsCaptured(Exception):
    pass


def _capture(csv_path, out_dir, **kwargs):
    raise _ArgsCaptured(kwargs)


_original_prepare = prep_mod.prepare
prep_mod.prepare = _capture
try:
    prep_mod.main(["--csv", "x.csv", "--out-dir", "x_out"])   # nothing is read or written
    raise RuntimeError("prepare() tidak dipanggil oleh main(); struktur script berubah.")
except _ArgsCaptured as captured:
    CLI_DEFAULTS = dict(captured.args[0])
finally:
    prep_mod.prepare = _original_prepare

PREPARE_DEFAULTS = {
    k: p.default for k, p in inspect.signature(prep_mod.prepare).parameters.items()
    if p.default is not inspect.Parameter.empty
}
print("\nDefault CLI (argparse)  :", CLI_DEFAULTS)
print("Default prepare()       :", PREPARE_DEFAULTS)
if CLI_DEFAULTS != PREPARE_DEFAULTS:
    raise RuntimeError("Default CLI berbeda dari default prepare(); periksa script sebelum melanjutkan.")
DOCUMENTED = {"val_frac": 0.1, "test_frac": 0.1, "seed": 42}
diff = {k: (CLI_DEFAULTS.get(k), v) for k, v in DOCUMENTED.items() if CLI_DEFAULTS.get(k) != v}
if diff:
    raise RuntimeError(f"Default berbeda dari dokumentasi (aktual, dokumen): {diff}")
print("Default sesuai dokumentasi (80/10/10, seed 42). neg_oversample =", CLI_DEFAULTS["neg_oversample"])

## 6. Preprocessing dan grouped split

Menjalankan script proyek apa adanya:

```
python scripts/prepare_data.py --csv <CSV_PATH> --out-dir /kaggle/working/data/processed_grouped_validation
```

- Parameter lain memakai default dari source code (lihat sel sebelumnya).
- Output berada di `/kaggle/working/data/`, di luar folder repository, sehingga tidak ikut ter-commit.
- Hanya `train/val/test.jsonl` lama dari run sebelumnya di folder output yang dihapus,
  agar validasi tidak membaca file basi.
- Seluruh stdout disimpan ke `/kaggle/working/prepare_data_validation_stdout.txt`.

In [ ]:
import ast
import re
import time

OUT_DIR = WORKING_DIR / "data" / "processed_grouped_validation"
STDOUT_LOG = WORKING_DIR / "prepare_data_validation_stdout.txt"
if REPO_DIR.resolve() in OUT_DIR.resolve().parents:
    raise RuntimeError("OUT_DIR tidak boleh berada di dalam repository.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILES = {s: OUT_DIR / f"{s}.jsonl" for s in SPLITS}
for path in OUTPUT_FILES.values():
    if path.exists():
        print("Hapus output lama:", path)
        path.unlink()

PREPARE_CMD = [sys.executable, str(PREPARE_SCRIPT), "--csv", str(CSV_PATH), "--out-dir", str(OUT_DIR)]
print("Perintah:", " ".join(f'"{a}"' if " " in a else a for a in PREPARE_CMD), "\n")
t0 = time.time()
prep_res = subprocess.run(PREPARE_CMD, cwd=REPO_DIR, capture_output=True,
                          text=True, encoding="utf-8", env=CHILD_ENV)
PREPARE_SECONDS = round(time.time() - t0, 1)
STDOUT_LOG.write_text(prep_res.stdout + ("\n[stderr]\n" + prep_res.stderr if prep_res.stderr else ""),
                      encoding="utf-8")
print(prep_res.stdout)
if prep_res.stderr:
    print("[stderr]\n" + prep_res.stderr)
if prep_res.returncode != 0:
    raise RuntimeError(f"prepare_data.py gagal (exit {prep_res.returncode}). Lihat stderr di atas.")
missing_out = [str(p) for p in OUTPUT_FILES.values() if not p.is_file()]
if missing_out:
    raise FileNotFoundError(f"Output tidak terbentuk: {missing_out}")
print(f"Selesai dalam {PREPARE_SECONDS} detik. Log: {STDOUT_LOG}")

# --- Parse the statistics printed by the script ---------------------------------
out_text = prep_res.stdout


def _grab(pattern, convert=int):
    m = re.search(pattern, out_text, re.M)
    if not m:
        raise RuntimeError(f"Pola {pattern!r} tidak ditemukan di stdout; format output script berubah.")
    return convert(m.group(1))


m = re.search(r"Loaded (\d+) reviews, (\d+) empty after clean_text -> (\d+) enter the split", out_text)
if not m:
    raise RuntimeError("Baris 'Loaded ...' tidak ditemukan di stdout; format output script berubah.")
CLI_STATS = {"n_loaded": int(m.group(1)), "n_empty_after_clean": int(m.group(2)),
             "n_after_clean": int(m.group(3))}
for key in ("n_rows_input", "n_groups", "n_conflict_groups", "n_rows_dropped", "n_rows_kept"):
    CLI_STATS[key] = _grab(rf"^\s*{key}\s*:\s*(\d+)")
CLI_STATS["split_rows"] = _grab(r"^\s*split_rows\s*:\s*(\{.*\})\s*$", ast.literal_eval)
CLI_STATS["written_sizes"] = _grab(r"^Written \(.*?\):\s*(\{.*\})\s*$", ast.literal_eval)
CLI_STATS["shared_review_texts"] = _grab(
    r"^Identical review texts shared across splits:\s*(\{.*\})\s*$", ast.literal_eval
)
print("\nStatistik yang diparse dari stdout:")
print(json.dumps(CLI_STATS, indent=2))

## 7. Validasi hasil

Semua pemeriksaan memakai fungsi dari repository (`clean_text`, `load_reviews`, `prepare`)
dan tidak menulis ulang logika split. Sumber data untuk setiap pemeriksaan:

| Pemeriksaan | Sumber |
|---|---|
| Statistik split lengkap (`split_stats`, `n_used`, `split_sizes`, ...) | `prepare()` dijalankan ulang di proses ini ke folder sementara, dengan default yang sama |
| Determinisme | hash JSONL run ulang harus identik dengan output CLI (folder sementara lalu dihapus) |
| Jumlah record per split | membaca `train/val/test.jsonl` hasil CLI |
| Kebocoran antar-split | `clean_text(review)` per record JSONL, lalu irisan antar-split |
| Conflict groups | kunci `clean_text` di input yang **tidak** muncul di output harus sama dengan `n_conflict_groups`, dan baris-barisnya sama dengan `n_rows_dropped` |
| Oversampling | val/test = `split_rows`; tambahan train = `(floor(f)-1)·n_neg + round(frac(f)·n_neg)` |

Catatan: `train.jsonl` sudah berisi salinan oversampling, jadi proporsi 80/10/10 dihitung dari
`split_rows`, yaitu **sebelum** oversampling.

In [ ]:
import math
import tempfile
from collections import Counter
from itertools import combinations

CHECKS = {}        # name -> bool (hard checks; all must pass)
WARNINGS = []


def check(name, ok, detail=""):
    CHECKS[name] = bool(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


def read_jsonl(path):
    def _no_nan(token):
        raise ValueError(f"konstanta JSON non-standar {token} di {path}")
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line, parse_constant=_no_nan) for line in fh if line.strip()]


# --- A. Full statistics + determinism: run prepare() again into a temp folder -----
with tempfile.TemporaryDirectory(dir=WORKING_DIR) as tmp:
    STATS = prep_mod.prepare(CSV_PATH, Path(tmp), **CLI_DEFAULTS)
    rerun_hash = {s: sha256_file(Path(tmp) / f"{s}.jsonl") for s in SPLITS}
cli_hash = {s: sha256_file(p) for s, p in OUTPUT_FILES.items()}
SPLIT_STATS = STATS["split_stats"]

print("== Statistik split (split_data_with_stats, sebelum oversampling) ==")
for key in ("n_rows_input", "n_groups", "n_conflict_groups", "n_rows_dropped", "n_rows_kept"):
    print(f"  {key:18s}: {SPLIT_STATS[key]:,}")
print(f"  {'split_rows':18s}: {SPLIT_STATS['split_rows']}")
print(f"  {'split_groups':18s}: {SPLIT_STATS['split_groups']}")
print(f"  {'split_frac':18s}: " + ", ".join(f"{k} {v:.4f}" for k, v in SPLIT_STATS["split_frac"].items()))
print(f"  conflict_examples : {len(SPLIT_STATS['conflict_examples'])} ditampilkan dari {SPLIT_STATS['n_conflict_groups']}")
for text in SPLIT_STATS["conflict_examples"]:
    print(f"    - {text!r}")
print()

check("stdout CLI == statistik prepare() (run ulang)",
      all(CLI_STATS[k] == (STATS[k] if k in STATS else SPLIT_STATS[k]) for k in CLI_STATS),
      "n_loaded, n_after_clean, n_conflict_groups, n_rows_dropped/kept, split_rows, written_sizes, shared")
check("deterministik: JSONL run ulang identik byte-per-byte", rerun_hash == cli_hash)

# --- B. Row bookkeeping ----------------------------------------------------------
check("n_after_clean == n_rows_input", STATS["n_after_clean"] == SPLIT_STATS["n_rows_input"])
check("n_rows_kept == n_rows_input - n_rows_dropped",
      SPLIT_STATS["n_rows_kept"] == SPLIT_STATS["n_rows_input"] - SPLIT_STATS["n_rows_dropped"])
check("n_used == n_rows_kept == sum(split_rows)",
      STATS["n_used"] == SPLIT_STATS["n_rows_kept"] == sum(SPLIT_STATS["split_rows"].values()))
check("split_sizes (sebelum oversampling) == split_rows", STATS["split_sizes"] == SPLIT_STATS["split_rows"])
check("n_loaded == n_after_clean + n_empty_after_clean",
      STATS["n_loaded"] == STATS["n_after_clean"] + STATS["n_empty_after_clean"])

# --- C. JSONL files: structure and record counts --------------------------------
records = {s: read_jsonl(p) for s, p in OUTPUT_FILES.items()}
written = {s: len(r) for s, r in records.items()}
bad_fields = {s: sum(set(r) != set(RECORD_FIELDS) for r in recs) for s, recs in records.items()}
check("semua record JSONL punya field RECORD_FIELDS", not any(bad_fields.values()), str(bad_fields))
check("jumlah record JSONL == written_sizes", written == STATS["written_sizes"], str(written))
check("val/test tidak di-oversample (record == split_rows)",
      all(written[s] == SPLIT_STATS["split_rows"][s] for s in SPLITS if s != "train"))

factor = CLI_DEFAULTS["neg_oversample"]
train_non_neg = sum(r["sentiment"] != "negative" for r in records["train"])
n_neg_pre = SPLIT_STATS["split_rows"]["train"] - train_non_neg          # negatives before oversampling
expected_extra = (int(factor) - 1) * n_neg_pre + int(round((factor - int(factor)) * n_neg_pre))
check(f"oversampling train sesuai faktor {factor}",
      written["train"] == SPLIT_STATS["split_rows"]["train"] + expected_extra,
      f"train {SPLIT_STATS['split_rows']['train']:,} + {expected_extra:,} salinan negatif = {written['train']:,}")

# --- D. Leakage across splits (same clean_text as the project) ------------------
not_clean = sum(clean_text(r["review"]) != r["review"] for recs in records.values() for r in recs)
check("review di JSONL sudah dalam bentuk clean_text", not_clean == 0, f"{not_clean} tidak sama")
keys = {s: {clean_text(r["review"]) for r in recs} for s, recs in records.items()}
SHARED = {f"{a}&{b}": len(keys[a] & keys[b]) for a, b in combinations(SPLITS, 2)}
check("tidak ada teks clean_text yang sama antar-split", not any(SHARED.values()), str(SHARED))
check("hasil kebocoran sama dengan laporan script", SHARED == STATS["shared_review_texts"])

# --- E. Conflict groups (A2) and same-label groups (B1) --------------------------
def _label(v):
    return "<NA>" if v is None else str(v)


labels_per_key = {}
for recs in records.values():
    for r in recs:
        labels_per_key.setdefault(clean_text(r["review"]), set()).add(_label(r["sentiment"]))
multi = sum(len(v) > 1 for v in labels_per_key.values())
check("setiap kelompok di output hanya punya satu sentimen (A2)", multi == 0, f"{multi} kelompok bercampur")

loaded = load_reviews(CSV_PATH)                         # project loader, read-only
input_keys = loaded["review"].map(clean_text)
input_keys = input_keys[input_keys != ""]               # empty-after-clean rows never enter the split
output_keys = set().union(*keys.values())
dropped_keys = set(input_keys) - output_keys
check("n_groups == jumlah kunci clean_text unik di input", SPLIT_STATS["n_groups"] == input_keys.nunique())
check("kelompok input yang hilang dari output == n_conflict_groups",
      len(dropped_keys) == SPLIT_STATS["n_conflict_groups"], f"{len(dropped_keys):,} kelompok")
check("baris kelompok yang hilang == n_rows_dropped",
      int(input_keys.isin(dropped_keys).sum()) == SPLIT_STATS["n_rows_dropped"])
check("conflict_examples termasuk kelompok yang dibuang", set(SPLIT_STATS["conflict_examples"]) <= dropped_keys)
check("tidak ada output yang berasal dari kunci di luar input", output_keys <= set(input_keys))

dup_rows_kept = int((input_keys[~input_keys.isin(dropped_keys)].value_counts() > 1).sum())
print(f"\nKelompok non-konflik dengan >1 baris (dipertahankan utuh dalam satu split, B1): {dup_rows_kept:,}")

# --- F. Proportions (before oversampling) ---------------------------------------
target = {"train": 1 - CLI_DEFAULTS["val_frac"] - CLI_DEFAULTS["test_frac"],
          "val": CLI_DEFAULTS["val_frac"], "test": CLI_DEFAULTS["test_frac"]}
prop = pd.DataFrame({
    "rows": SPLIT_STATS["split_rows"],
    "groups": SPLIT_STATS["split_groups"],
    "frac": SPLIT_STATS["split_frac"],
    "target": target,
    "records_jsonl": written,
})
prop["frac_minus_target"] = prop["frac"] - prop["target"]
print("\n== Proporsi split (sebelum oversampling) ==")
print(prop.to_string(float_format=lambda v: f"{v:.4f}"))
for s in SPLITS:
    if abs(prop.loc[s, "frac_minus_target"]) > 0.01:
        WARNINGS.append(f"proporsi {s} menyimpang > 1 poin persen dari target ({prop.loc[s, 'frac']:.4f})")

# Sentiment per split before oversampling: val/test straight from JSONL,
# train = JSONL non-negatives + negatives before oversampling.
SENT_BEFORE = {s: Counter(_label(r["sentiment"]) for r in records[s]) for s in SPLITS if s != "train"}
SENT_BEFORE["train"] = Counter(_label(r["sentiment"]) for r in records["train"] if r["sentiment"] != "negative")
SENT_BEFORE["train"]["negative"] = n_neg_pre
sent = pd.DataFrame(SENT_BEFORE).fillna(0).astype(int)[list(SPLITS)]
print("\n== Sentimen per split (sebelum oversampling) ==")
print(pd.concat({"jumlah": sent, "persen": (100 * sent / sent.sum()).round(2)}, axis=1).to_string())
print("\n== Sentimen train.jsonl (setelah oversampling) ==")
print(pd.Series(Counter(_label(r["sentiment"]) for r in records["train"])).to_string())

# --- Result -----------------------------------------------------------------------
print()
for w in WARNINGS:
    print("[WARN]", w)
failed = [k for k, ok in CHECKS.items() if not ok]
print(f"\n{len(CHECKS) - len(failed)}/{len(CHECKS)} pemeriksaan lulus.")
if failed:
    raise AssertionError("Pemeriksaan gagal:\n- " + "\n- ".join(failed))

## 8. Ringkasan eksperimen

Ringkasan disusun dari variabel hasil eksekusi di atas (bukan diketik manual), lalu disimpan ke
`/kaggle/working/prepare_data_validation_summary.json`. Hash CSV dicek ulang untuk memastikan
dataset asli tidak berubah.

In [ ]:
from datetime import datetime, timezone

CSV_SHA256_AFTER = sha256_file(CSV_PATH)
check("CSV asli tidak berubah (SHA-256 sama)", CSV_SHA256_AFTER == CSV_SHA256_BEFORE)
SUMMARY_PATH = WORKING_DIR / "prepare_data_validation_summary.json"


def _versions():
    out = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__}
    try:
        import torch
        out["torch"] = torch.__version__
    except ImportError:
        out["torch"] = None
    return out


summary = {
    "experiment": "01_prepare_data_validation",
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "repository": {"url": REPO_URL, "branch": BRANCH, "commit": REPO_COMMIT},
    "dataset": {
        "csv_path": str(CSV_PATH),
        "sha256": CSV_SHA256_BEFORE,
        "unchanged_after_run": CSV_SHA256_AFTER == CSV_SHA256_BEFORE,
        "n_rows_csv": N_RAW_ROWS,
        "columns": RAW_COLUMNS,
    },
    "command": PREPARE_CMD,
    "config": CLI_DEFAULTS,
    "runtime_seconds_prepare_data": PREPARE_SECONDS,
    "counts": {
        "n_rows_csv": N_RAW_ROWS,
        "n_loaded": STATS["n_loaded"],
        "n_empty_after_clean": STATS["n_empty_after_clean"],
        "n_after_clean": STATS["n_after_clean"],
        "n_groups": SPLIT_STATS["n_groups"],
        "n_conflict_groups": SPLIT_STATS["n_conflict_groups"],
        "n_rows_dropped": SPLIT_STATS["n_rows_dropped"],
        "n_rows_kept": SPLIT_STATS["n_rows_kept"],
        "n_used": STATS["n_used"],
    },
    "split": {
        "split_rows_before_oversampling": SPLIT_STATS["split_rows"],
        "split_groups": SPLIT_STATS["split_groups"],
        "split_frac": SPLIT_STATS["split_frac"],
        "target_frac": target,
        "records_written_jsonl": written,
        "negatives_train_before_oversampling": n_neg_pre,
        "negatives_added_by_oversampling": expected_extra,
        "sentiment_before_oversampling": {s: dict(SENT_BEFORE[s]) for s in SPLITS},
    },
    "leakage": {
        "shared_clean_texts": SHARED,
        "no_leakage": not any(SHARED.values()),
    },
    "conflict_examples": SPLIT_STATS["conflict_examples"],
    "checks": CHECKS,
    "warnings": WARNINGS,
    "all_checks_passed": all(CHECKS.values()),
    "outputs": {
        **{f"{s}_jsonl": str(p) for s, p in OUTPUT_FILES.items()},
        "stdout_log": str(STDOUT_LOG),
        "summary_json": str(SUMMARY_PATH),
    },
    "environment": _versions(),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

c, sp = summary["counts"], summary["split"]
print("RINGKASAN EKSPERIMEN — ReviewLens GPT · prepare data + grouped split")
print("=" * 70)
print(f"Dataset sumber           : {CSV_PATH}")
print(f"Commit repository        : {REPO_COMMIT[:12]} ({BRANCH})")
print(f"Baris CSV mentah         : {c['n_rows_csv']:,}")
print(f"Setelah load_reviews     : {c['n_loaded']:,}  (buang review kosong & duplikat (item_id, review))")
print(f"Kosong setelah clean_text: {c['n_empty_after_clean']:,}")
print(f"Masuk split (n_after_clean): {c['n_after_clean']:,}  dalam {c['n_groups']:,} kelompok")
print(f"Conflict groups          : {c['n_conflict_groups']:,}")
print(f"Baris dibuang            : {c['n_rows_dropped']:,}")
print(f"Baris dipertahankan      : {c['n_rows_kept']:,}  (n_used)")
print("Record per split (sebelum oversampling):")
for s in SPLITS:
    print(f"  {s:5s}: {sp['split_rows_before_oversampling'][s]:>7,}  ({sp['split_frac'][s]:.2%})"
          f"  -> ditulis ke JSONL: {sp['records_written_jsonl'][s]:,}")
print(f"Kebocoran antar-split    : {SHARED}  -> {'TIDAK ADA' if summary['leakage']['no_leakage'] else 'ADA!'}")
print(f"CSV asli tidak berubah   : {summary['dataset']['unchanged_after_run']}")
print(f"Pemeriksaan              : {sum(CHECKS.values())}/{len(CHECKS)} lulus"
      + (f"; peringatan: {WARNINGS}" if WARNINGS else ""))
print("Lokasi output:")
for k, v in summary["outputs"].items():
    print(f"  {k:12s}: {v}")
if not summary["all_checks_passed"]:
    raise AssertionError("Ada pemeriksaan yang gagal; lihat 'checks' di ringkasan JSON.")

### Langkah berikutnya

- Jika semua pemeriksaan lulus, `train/val/test.jsonl` di
  `/kaggle/working/data/processed_grouped_validation/` siap dipakai untuk tahap berikutnya.
- Vocabulary (`WordTokenizer.build` → `vocab.json`) dibangun di notebook terpisah **hanya** dari `train.jsonl`.
- File hasil preprocessing dan ringkasan JSON tidak di-commit ke GitHub (dataset turunan; `/data/` juga ada di `.gitignore`).
  Simpan sebagai output notebook Kaggle atau sebagai Kaggle Dataset jika dibutuhkan di notebook lain.